# Lab 6: Describe the reviews
Same Airbnb table as in class, but this time I describe `number_of_reviews` (Part 1) and the rating (Part 2).

In [ ]:
import pandas as pd
import numpy as np

# change the list if your file has other date columns
listings = pd.read_csv('data/listings_clean.csv',
                       parse_dates=['last_scraped', 'first_review', 'last_review'])
print(listings.shape)      # (4574, ...)
reviews = listings['number_of_reviews']

# Part 1: Practise

## Task 1: The mean and the median

In [ ]:
small = pd.Series([3, 5, 8, 10, 12])
print('Small list:    mean', small.mean(), ' median', small.median())      # 7.6 and 8.0

with_big = pd.concat([small, pd.Series([1293])], ignore_index=True)
print('With one 1293: mean', round(with_big.mean(), 1), ' median', with_big.median())   # 221.8 and 9.0

print('Real column:   mean', round(reviews.mean(), 1), ' median', reviews.median())      # 35.8 and 9.0
# One listing with 1293 reviews made the mean jump from 7.6 to 221.8, but the median only moved from 8 to 9.
# The mean is pulled by extreme values, the median is not.
# For "how many reviews does a typical listing have?" I use the median (9): half of all listings
# have 9 reviews or fewer. The mean (35.8) is dragged up by a few very popular listings.

## Task 2: Percentiles and the spread

In [ ]:
reviews.describe().round(1)
# count 4574, mean 35.8, std 78.4, min 0, 25% 2, 50% 9, 75% 29, max 1293

In [ ]:
reviews.quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99])
# 10% = 0, 25% = 2, 50% = 9, 75% = 29, 90% = 94, 99% = 400

q1 = reviews.quantile(0.25)
q3 = reviews.quantile(0.75)
iqr = q3 - q1
print(q1, q3, iqr)      # 2.0 29.0 27.0
# The IQR is 27: the middle half of the listings have between 2 and 29 reviews, so most of the
# listings are in a small range, even though the maximum is 1293.
# The 10th percentile is 0, so at least 10 % of the listings have no reviews at all.

## Task 3: The fences and the outliers

In [ ]:
upper = q3 + 1.5 * iqr
lower = q1 - 1.5 * iqr
print(upper, lower)                         # 69.5 and -38.5
# The lower fence is negative, and a number of reviews cannot be below 0, so there are no outliers
# at the bottom. All the outliers are at the top.

above = listings[reviews > upper]
print(len(above), round(len(above) / len(listings) * 100, 1))   # 582 listings, 12.7 %
print(above['room_type'].value_counts())    # Entire home/apt 446, Private room 121, Shared room 14, Hotel room 1

In [ ]:
print('Median price, outliers:   ', round(above['price'].median(), 1))                   # 1459.5
print('Median price, all the others:', listings[reviews <= upper]['price'].median())   # 1867.0
# The listings with the most reviews are cheaper than the rest. Cheaper places probably get booked
# more often, and every guest can leave a review. (The outliers are mostly entire homes, but that is
# also the biggest group in the whole table.)

## Task 4: Mark them, and compare

In [ ]:
# I do not remove the outliers, I only mark them.
listings['reviews_outlier'] = reviews > upper
print(listings['reviews_outlier'].sum())     # 582

without = listings[~listings['reviews_outlier']]['number_of_reviews']
table = pd.DataFrame({'all': reviews.agg(['count', 'mean', 'median', 'std']),
                      'without': without.agg(['count', 'mean', 'median', 'std'])}).round(1)
table['change %'] = ((table['without'] / table['all'] - 1) * 100).round(0)
table
# count 4574 -> 3992, mean 35.8 -> 12.9 (-64 %), median 9.0 -> 7.0 (-22 %), std 78.4 -> 15.5 (-80 %).
# The standard deviation changes the most, the median the least. The median hardly cares about outliers.

## Task 5: Describe the groups

In [ ]:
listings.groupby('room_type')['number_of_reviews'].describe().round(0)
# Entire home/apt: count 3760, median 8. Private room: count 787, median 12.
# Shared room: count 24, median 82. Hotel room: count 3, median 56.
# Shared rooms have the highest median among the groups that have enough listings.
# I do not want to say anything about hotel rooms: there are only 3 listings.

## Task 6: Correlation

In [ ]:
cols = ['price', 'accommodates', 'bedrooms', 'availability_365',
        'review_scores_rating', 'minimum_nights', 'days_since_review']
listings[['number_of_reviews'] + cols].corr()['number_of_reviews'].drop('number_of_reviews').round(2)
# price -0.09, accommodates -0.05, bedrooms -0.14, availability_365 0.18,
# review_scores_rating -0.04, minimum_nights -0.07, days_since_review -0.19
# All connections are weak. The strongest is days_since_review (-0.19): more reviews go together with
# fewer days since the last review. The weakest is review_scores_rating (-0.04).
# Negative: all of them except availability_365 (which is the only positive one).

In [ ]:
with_all = listings['number_of_reviews'].corr(listings['availability_365']).round(2)
no_out = listings[~listings['reviews_outlier']]
without_out = no_out['number_of_reviews'].corr(no_out['availability_365']).round(2)
print('With outliers:', with_all, ' without:', without_out)     # 0.18 and 0.0
# Without the outliers the connection is gone (0.18 becomes 0.00). The weak connection came only from
# the few listings with extreme numbers, so the first number should not be trusted.

# Part 2: Go deeper

## Task 1: Mean or median for the rating?

In [ ]:
rating = listings['review_scores_rating']       # missing for 644 listings
print('mean', round(rating.mean(), 2), ' median', round(rating.median(), 2), ' std', round(rating.std(), 2))
# mean 4.82, median 4.92, std 0.32
rating.quantile([0.25, 0.5, 0.75])

In [ ]:
# The mean (4.82) is lower than the median (4.92), the opposite of price. The tail goes to the LEFT:
# most listings are very close to the maximum of 5.0, and a few low ratings pull the mean down.
# It cannot go to the right because 5.0 is the top of the scale.
# I would report the median, 4.92, and write next to it that most listings are above 4.75
# (the 25th percentile), and that the scale is squeezed at the top: nearly everything is 4.5 to 5.

## Task 2: Outliers at the bottom

In [ ]:
q1_r, q3_r = rating.quantile(0.25), rating.quantile(0.75)
iqr_r = q3_r - q1_r
lower_r = q1_r - 1.5 * iqr_r
print(q1_r, q3_r, iqr_r, lower_r)           # 4.75 5.0 0.25 4.375

listings['rating_outlier'] = rating < lower_r      # NaN is not below anything, so missing ratings are False
print(listings['rating_outlier'].sum(), 'of', rating.notna().sum())    # 211 of 3930

In [ ]:
rated = listings[rating.notna()]
print(rated.groupby('rating_outlier')['number_of_reviews'].median())            # False 13, True 4
print((rated['number_of_reviews'] <= 5).groupby(rated['rating_outlier']).mean().mul(100).round(0))   # 28 %, 59 %
print(rated[rated['rating_outlier']]['room_type'].value_counts())    # Entire home/apt 160, Private room 51

print('Mean with:', round(rating.mean(), 2), ' without:', round(rating[~listings['rating_outlier']].mean(), 2))     # 4.82 -> 4.87
print('Median with:', round(rating.median(), 2), ' without:', round(rating[~listings['rating_outlier']].median(), 2))   # 4.92 -> 4.93
# The low-rated listings have fewer reviews (median 4 against 13), and 59 % of them have 5 reviews or fewer.
# So often one unhappy guest decides the rating.
# Decision: I MARK them (rating_outlier) and KEEP them. They are real listings and real opinions, not typing
# errors. Removing them would make the ratings look better than they are. The mean would go from 4.82 to 4.87.

## Task 3: Do superhosts get more reviews?

In [ ]:
# (a) number of listings and median reviews
listings.groupby('host_is_superhost')['number_of_reviews'].agg(['count', 'median'])
# Superhosts: 737 listings, median 53. Others: 3837 listings, median 7.

In [ ]:
# (b) how many years has each listing existed?
listings['years_existed'] = (listings['last_scraped'] - listings['first_review']).dt.days / 365.25
listings.groupby('host_is_superhost')['years_existed'].median().round(2)
# Superhosts 3.31 years, others 2.89 years. Only 0.4 years more, which cannot explain a difference of 7 times.

In [ ]:
# (c) reviews in the last 12 months
listings.groupby('host_is_superhost')['number_of_reviews_ltm'].median()
# Superhosts 16, others 1. The difference is large also in the last year.
# I CAN say: superhosts have many more reviews, also recently, and not only because they are older.
# I CANNOT say: that being a superhost makes you get more reviews. As far as I know, Airbnb gives the badge
# for a good record: enough bookings, a high rating, and fast answers. So a host needs many stays to become
# a superhost, and the many reviews can come BEFORE the badge. It can be the other way round.
# (I should check the criteria on Airbnb's page.)

## Task 4: My decisions about the outliers

**Reviews (`number_of_reviews`)**
- **What I did:** I marked the outliers in a new column, `reviews_outlier`, and kept them. 582 listings (12.7 %) are above the upper fence of 69.5. There are no outliers at the bottom, because the lower fence (-38.5) is below zero.
- **Why:** These are real, very popular listings and not typing mistakes. Removing them would hide the most popular part of the market.
- **What it does to the numbers:** Without them, the mean falls from 35.8 to 12.9 (-64 %), the standard deviation from 78.4 to 15.5 (-80 %), and the median only from 9 to 7 (-22 %).
- **What I report:** The median, **9 reviews**, together with the IQR, **2 to 29**: half of the listings have 9 reviews or fewer, and the middle half lies between 2 and 29. I do not report the mean on its own.

**Rating (`review_scores_rating`)**
- **What I did:** I marked the listings below the lower fence (4.375) in `rating_outlier` and kept them. That is 211 of the 3930 rated listings. 644 listings have no rating at all, and I left those as missing.
- **Why:** They are real opinions, and many of them have few reviews (median 4), so one unhappy guest decides the rating. Removing them would make the ratings look better than they are.
- **What it does to the numbers:** Without them the mean goes from 4.82 to 4.87 and the median from 4.92 to 4.93, which is a very small change.
- **What I report:** The median, **4.92**, and I write next to it that it is based on 3930 listings, that most listings are above 4.75, and that the scale is squeezed at the top, so small differences do not say much.

## Task 5: Your own question

In [ ]:
# Question: Do listings in the four central neighbourhoods have more reviews than listings outside the centre?
central = ['Södermalms', 'Östermalms', 'Norrmalms', 'Kungsholmens']
listings['is_central'] = listings['neighbourhood_cleansed'].isin(central)
listings.groupby('is_central')['number_of_reviews'].agg(['count', 'median', 'mean']).round(1)
# This does NOT tell me WHY. Central listings may simply be older, or cheaper, or smaller, and I did not check.
# The mean is pulled by outliers, so I look at the median. Reviews count guests who wrote something,
# not all the guests, and I do not know how long each listing has existed.
# (Replace this with a question of your own!)